# Cycle 3 — Preprocessing: Player Injury Risk

**dataset** player_injuries_processed.csv`

This notebook takes the raw `players_injured.csv` and prepares it for machine learning. Every issue identified during exploration is addressed here step by step. The output is a clean, model-ready dataset saved to `data/processed/`.

In [1]:
import sys, os  

_here = os.getcwd()                                       
while not os.path.isdir(os.path.join(_here, 'data')):     
    _p = os.path.dirname(_here)                           
    if _p == _here: raise RuntimeError('project root not found')  
    _here = _p
if _here not in sys.path:
    sys.path.insert(0, _here)                  

from config import Paths, ensure_dirs
ensure_dirs()  


In [ ]:
# LOADING
import pandas as pd 
import numpy as np 
import os          
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(str(Paths.PLAYER_INJURIES_RAW))  
print(f'Raw shape: {df.shape}') 


Raw shape: (1301, 30)


## Construct Target Variable 

Creating the binary target `High_Injury` 
- 1 = player missed ≥ 28 days this season
- 0 = fewer than 28 days

In [3]:
THRESHOLD = 28                                    # 28 days = ~1 month — meaningful injury threshold
df['High_Injury'] = (df['season_days_injured'] >= THRESHOLD).astype(int)  # binary: 1=high injury burden
counts = df['High_Injury'].value_counts()
print(f'Target: High_Injury (season_days_injured >= {THRESHOLD} days)')
print(f'  Low Injury  (0): {counts[0]} ({counts[0]/len(df)*100:.1f}%)')  # ~29%
print(f'  High Injury (1): {counts[1]} ({counts[1]/len(df)*100:.1f}%)')  # ~71%


Target: High_Injury (season_days_injured >= 28 days)
  Low Injury  (0): 388 (29.8%)
  High Injury (1): 913 (70.2%)


## Drop Post-Season Leakage and Identifier Columns

Removes columns that are either identifiers (player ID, date of birth, nationality) or post-season measurements that would only be known after the season ends.

In [4]:
drop_cols = [
    # Identifiers — no predictive value as features
    'p_id2', 'dob', 'nationality', 'work_rate', 'position', 'start_year',
    # Post-season leakage — only known after the season ends
    'season_days_injured', 'total_days_injured',
    'season_minutes_played', 'season_games_played', 'season_matches_in_squad',
    'total_minutes_played', 'total_games_played',
]
df = df.drop(columns=drop_cols)  # remove all at once
print(f'After dropping leakage/identifiers: {df.shape}')   # confirm column reduction
print(f'Remaining columns: {list(df.columns)}')


After dropping leakage/identifiers: (1301, 18)
Remaining columns: ['height_cm', 'weight_kg', 'pace', 'physic', 'fifa_rating', 'age', 'cumulative_minutes_played', 'cumulative_games_played', 'minutes_per_game_prev_seasons', 'avg_days_injured_prev_seasons', 'avg_games_per_season_prev_seasons', 'bmi', 'work_rate_numeric', 'position_numeric', 'significant_injury_prev_season', 'cumulative_days_injured', 'season_days_injured_prev_season', 'High_Injury']


## Fill First-Season NaN with 0

Players in their first tracked season have no prior injury history. NaN means 0 history, not unknown — filling with 0 is correct.

In [5]:
# Columns that are NaN for first-season players (no prior history exists yet)
history_cols = [
    'cumulative_minutes_played', 'cumulative_games_played',
    'minutes_per_game_prev_seasons', 'avg_days_injured_prev_seasons',
    'avg_games_per_season_prev_seasons', 'significant_injury_prev_season',
    'cumulative_days_injured', 'season_days_injured_prev_season'
]
before = df[history_cols].isnull().sum().sum()  # count NaN before
df[history_cols] = df[history_cols].fillna(0)   # fill with 0 = 'no history'
after = df[history_cols].isnull().sum().sum()   # should be 0
print(f'Filled {before} NaN values in history columns with 0')
print(f'Remaining nulls in history cols: {after}')  # confirm zero


Filled 4844 NaN values in history columns with 0
Remaining nulls in history cols: 0


## Impute pace, physic with Median; position_numeric with Mode

Fills remaining NaN values in `pace`, `physic`, and `position_numeric` using median (for continuous) and mode (for categorical).

In [5]:
for col in ['pace', 'physic']:         # FIFA attributes missing for unrated players
    median_val = df[col].median()        # use median — robust to outliers
    df[col] = df[col].fillna(median_val)
    print(f'  {col}: filled {df[col].isnull().sum()} NaN with median={median_val:.1f}')

mode_pos = df['position_numeric'].mode()[0]  # most common position code
df['position_numeric'] = df['position_numeric'].fillna(mode_pos)  # mode imputation for ordinal
print(f'  position_numeric: filled NaN with mode={mode_pos}')


  pace: filled 0 NaN with median=71.0
  physic: filled 0 NaN with median=72.3
  position_numeric: filled NaN with mode=1.0


## Final Validation & Save

In [7]:
print('Null check:')
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else '  0 nulls remaining')  # must be zero
print()
print(f'Final shape: {df.shape}')                                        # rows × (features + target)
print(f'Features: {df.shape[1]-1} | Target: High_Injury')
print()
print(df.describe().round(2))                                            # summary stats for all columns

os.makedirs(str(Paths.DATA_PROCESSED), exist_ok=True)                   # create folder if missing
df.to_csv(str(Paths.PLAYER_INJURIES_PROCESSED), index=False)            # save without row index
print('\nSaved: ../data/processed/player_injuries_processed.csv')


Null check:
  0 nulls remaining

Final shape: (1301, 18)
Features: 17 | Target: High_Injury

       height_cm  weight_kg     pace   physic  fifa_rating      age  \
count    1301.00    1301.00  1301.00  1301.00      1301.00  1301.00   
mean      182.52      76.83    70.37    70.95        74.51    26.64   
std         6.82       7.36    10.49     7.88         5.78     3.94   
min       163.00      58.00    28.33    40.75        53.00    17.00   
25%       178.00      72.00    64.17    67.40        71.33    24.00   
50%       183.00      76.00    71.00    72.33        75.17    27.00   
75%       187.67      82.00    77.50    76.00        78.50    29.00   
max       203.00      99.00    93.00    88.17        89.50    39.00   

       cumulative_minutes_played  cumulative_games_played  \
count                    1301.00                  1301.00   
mean                     2226.93                    28.54   
std                      4253.83                    53.28   
min                    